In [ ]:
include( "../model.jl" )
include( "../cluster.jl" )
include( "../prob.jl" )

include( "../0-params/rho-list.jl" )
include( "../0-params/spd-list.jl" )
include( "../0-params/nos-list.jl" )

using Optim
using ThreadsX

home, base = expfolder( "system" )

In [ ]:
# Hyper parameters.
N = 1024;  s = 0.251
println( "Speed: ", s )
println( "Density: ", ρlist )

# Create folder and extract critical noise.
folderlist = [findfolder( N, ρ, s; base=base, crit=true ) for ρ ∈ ρlist]

# Extract critical noise values.
ηclist = first.( readdlm.( folderlist.*"../order-0.5.txt" ) );

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ, "Importing data."; dt=1.0 )

# Initialize data variables.
κlist = BitVector( undef, Nρ )
Fdata = Vector{Vector{defInt}}( undef, Nρ )
δFdata = Vector{Matrix{defInt}}( undef, Nρ )

# Import cluster dynamical data.
for i ∈ 1:Nρ
    κlist[i] = ispath( folderlist[i]*"frequency-map.txt" )
    Fdata[i]  = !κlist[i] ? fill( 0, N ) :
        round.( defInt, vec( readdlm( folderlist[i]*"cluster-frequency.txt" ) ) )
    δFdata[i] = !κlist[i] ? fill( 0, N, N ) :
        round.( defInt, readdlm( folderlist[i]*"frequency-map.txt" ) )
    next!( meter )
end

# Extract the clusters which can be optimized over for each parameter combination.
ιdata = [isnothing( δFdata[i] ) ? nothing : 0 .< vec( sum( δFdata[i]; dims=2 ) ) for i ∈ 1:Nρ];

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ*N, "Solving for exponents."; dt=1.0 )

# Initialize data variables.
λlist = Vector{defFloat}( undef, Nρ )
θdata = [Matrix{defFloat}( undef, N, 3 ) for _ ∈ 1:Nρ]

# Solve for the parameters.
θ0 = [1.5, 1.5, 0.66]
@threads for i ∈ 1:Nρ
    # Calibrate power-law on the cluster mass distribution.
    λlist[i] = 10^optimize( createzipfloss( N, Fdata[i] ),  [log10( 1.5 )] ).minimizer[1]

    # For each measured cluster size.
    @threads for j ∈ 1:N
        # Run calibration for each cluster size.
        θdata[i][j,:] = ιdata[i][j] ? convert( runopt( j, δFdata[i], N, θ0 ).minimizer ) : fill( NaN, 3 )

        # Update progress meter.
        next!( meter )
    end
end

In [ ]:
i = 18

# Plot the optimal rates for changing cluster size.
plt = plot(; layout=(3,1), size=(400,600), legend=false )

for j ∈ 1:2
    scatter!( plt[j], 1:N, θdata[i][:,j]; color=:gray )
    plot!( plt[j], [1,N], fill( λlist[i], 2 ); color=:black, linestyle=:dash, lw=2 )
    plot!( plt[j]; xlims=(1,N), ylims=(0,3) )
end

scatter!( plt[3], 1:N, θdata[i][:,3]; color=:gray )
plot!( plt[3]; xlims=(1,N), ylims=(0,1) )

In [ ]:
# Save parameter data.
if false
    # Save the cluster change parameters for each parameter case.
    for i ∈ 1:Nρ
        writedlm( folderlist[i]*"power-law.txt", λlist[i] )
        writedlm( folderlist[i]*"optimal-rates.txt", θdata[i] )
    end
end